In [ ]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
from tqdm import tqdm
import datetime as dt

In [ ]:
dtm_now = dt.datetime.today()
print(f'Latest run date: {dtm_now}')

### Functions

In [ ]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [ ]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'
# dict
list_str_replace = [
    '15_in_60',
    '30_in_90',
    '30_in_180',
    '30_in_360',
    '60_in_720',
]

### Output directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### DB Connection

In [ ]:
# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

### Get DPD targets

In [ ]:
# read query
str_filepath = './sql/query_dpd.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

In [ ]:
list_df = []
for str_replace in tqdm(list_str_replace):
    # DPD
    STRDPD = str_replace.split('_in_')[0]
    # total
    STRTOTAL = str_replace.split('_in_')[1]
    # DPD minus 1
    STRDPDMINUS1 = str(int(STRDPD) - 1)
    
    # replace
    str_query_tmp = str_query.replace('STRDPD', STRDPD)
    str_query_tmp = str_query_tmp.replace('STRTOTAL', STRTOTAL)
    str_query_tmp = str_query_tmp.replace('DPDMINUS1', STRDPDMINUS1)
    
    # pull from db
    df = pd.read_sql_query(
        str_query_tmp, 
        con=conn,
    )
    # set index
    df = df.set_index('bigAccountId')
    # append
    list_df.append(df)

# join
df = pd.concat(list_df, axis=1, join='outer')
# reset index
df.reset_index(inplace=True)
# show
df

### Close connection

In [ ]:
# close
conn.close()

### Make run date column to get days on books

In [ ]:
df['run_date'] = dtm_now
# show
df

### Save

In [ ]:
%%time

# save
str_filename = 'df_targets.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

### Upload to s3

In [ ]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

### Clean-up

In [ ]:
os.remove(str_local_path)